# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:**

Cada imagen tiene $32 \times 32 = 1024$ píxeles, así que después de `Flatten()` queda un vector de **1024 valores**. Ya no hay canal de color: la conversión a gris pasó de $32\times32\times3 = 3072$ valores a 1024.


## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())


In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()

In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:**

Porque una capa `Dense` calcula $\mathbf{z} = W\mathbf{x} + \mathbf{b}$, es decir, espera un **vector** por ejemplo. La imagen llega como una matriz de forma `(32, 32)`, y `Flatten()` la reorganiza en un vector de forma `(1024,)` poniendo las filas una detrás de otra. No aprende nada (tiene 0 parámetros), solo cambia la forma del tensor.

**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:**

Porque hay **10 clases** en CIFAR10. Cada neurona produce el logit (puntaje sin normalizar) de una clase, y la predicción es la clase con el logit más alto.

**4.** ¿Qué función cumple `ReLU`?

**Respuesta:**

`ReLU`, $f(z) = \max(0, z)$, introduce la **no linealidad**. Sin ella, dos capas `Dense` seguidas serían equivalentes a una sola transformación lineal ($W_2(W_1 x + b_1) + b_2 = W x + b$) y la red no podría aprender fronteras de decisión complejas. Además es barata de calcular y su derivada es 1 para entradas positivas, lo que ayuda a que el gradiente no se desvanezca.


---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:**

Porque la última capa no tiene Softmax y entrega **logits**, no probabilidades. Con `from_logits=True` la función de pérdida aplica internamente el Softmax y el logaritmo juntos, usando el truco de log sum exp, lo que es numéricamente más estable que calcular primero probabilidades muy pequeñas y luego su logaritmo. Si la dejáramos en `False`, Keras trataría los logits como si ya fueran probabilidades y la pérdida sería incorrecta.


---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)

### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:**

Una época es **una pasada completa por todo el conjunto de entrenamiento**: el modelo ve una vez cada una de las 50 000 imágenes. Con `epochs=10` el modelo recorre el dataset 10 veces.

**7.** ¿Qué representa `batch_size=64`?

**Respuesta:**

Que el gradiente se calcula y los pesos se actualizan con grupos (mini batches) de **64 imágenes** a la vez. Así, una época tiene $\lceil 50000 / 64 \rceil = 782$ actualizaciones de los parámetros, y en total el entrenamiento hace unas 7820. Es un compromiso entre usar una sola imagen (gradiente muy ruidoso) y todo el dataset (cada paso muy costoso).


## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:**

Hay **evidencia de un overfitting leve que va creciendo**. El accuracy de entrenamiento sube de forma sostenida durante las 10 épocas, mientras que el de validación sube al principio y luego se estanca alrededor de 0.40 con oscilaciones. La brecha entre ambas curvas aumenta en las últimas épocas, y en la gráfica de pérdida la de validación deja de bajar mientras la de entrenamiento sigue disminuyendo. No es un overfitting severo (las curvas no se separan mucho), pero indica que la red empieza a memorizar detalles del entrenamiento que no generalizan. *Nota:* aquí se usa el conjunto de test como validación, así que en rigor no sería una estimación independiente.


Para apoyar la respuesta, la siguiente celda muestra también las curvas de pérdida y la brecha entre entrenamiento y validación en cada época.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida")
plt.legend()
plt.show()

for e, (a, va) in enumerate(zip(history.history["accuracy"], history.history["val_accuracy"]), 1):
    print(f"Epoch {e:2d}: train = {a:.3f}  val = {va:.3f}  brecha = {a - va:+.3f}")

---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:**

**Mucho más difícil.** Con la misma receta en Fashion MNIST se obtiene cerca de 88 % de accuracy en test, mientras que aquí el modelo queda alrededor de **40 %**. Sigue estando muy por encima del azar (10 %), así que algo aprende, pero comete errores en la mayoría de las imágenes.

La razón es la naturaleza de las imágenes. Fashion MNIST tiene prendas centradas, con fondo negro uniforme, siempre en la misma posición y escala. CIFAR10 tiene fotografías naturales: el objeto aparece en distintas posiciones, tamaños y ángulos, con fondos variados, y además al pasar a gris perdemos el color, que es una pista importante (cielo azul para aviones, agua para barcos, verde para ranas). Una red densa que mira píxeles individuales no tiene forma de reconocer "un gato" si aparece en otra posición de la imagen.


---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])


In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:**

Porque Softmax está definido como

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{10} e^{z_j}}$$

Cada término es positivo y todos se dividen por la misma suma, de modo que $\sum_i p_i = 1$ por construcción. Así la salida es una distribución de probabilidad sobre las 10 clases, mutuamente excluyentes. Decimos *aproximadamente* porque los cálculos se hacen en `float32` y los errores de redondeo pueden dar algo como 0.99999994 o 1.0000001.


---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


### Accuracy por clase y confusiones más frecuentes

Esta celda adicional apoya la pregunta 2 de la reflexión.

In [ ]:
all_logits = model.predict(X_test, batch_size=256, verbose=0)
all_preds = all_logits.argmax(axis=1)

print("Accuracy por clase:")
for k, name in enumerate(class_names):
    mask = y_test == k
    print(f"  {name:12s}: {np.mean(all_preds[mask] == k):.3f}")

cm = np.zeros((10, 10), dtype=int)
for t, p in zip(y_test, all_preds):
    cm[t, p] += 1
off = cm.copy()
np.fill_diagonal(off, 0)
print("\nConfusiones más frecuentes (real → predicha):")
for idx in np.argsort(off, axis=None)[::-1][:6]:
    t, p = divmod(idx, 10)
    print(f"  {class_names[t]:10s} → {class_names[p]:10s}: {off[t, p]} imágenes")

---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta:**

El accuracy en test fue de aproximadamente **40 %** (el valor exacto aparece en la celda de `evaluate()`). Es casi 4 veces mejor que el azar, pero bajo para un clasificador de imágenes.

### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:**

Según el accuracy por clase y las confusiones más frecuentes, las clases más difíciles son las de **animales**, en especial **cat, dog, bird y deer**: tienen formas y texturas parecidas, aparecen en muchas poses y sobre fondos naturales similares. Gato y perro son la confusión clásica. También hay confusión entre **automobile y truck** y entre **airplane y ship**; estas últimas se separaban en buena parte por el color del fondo (cielo o mar), información que se perdió con la escala de grises. Las clases con objetos de forma más rígida y característica, como ship, truck o automobile, suelen tener mejores resultados.

### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante sobre la imagen podría perderse al tratarla únicamente como un vector?

**Respuesta:**

Se pierde la **estructura espacial**. Al aplanar, la red no sabe qué píxeles son vecinos: un píxel y el que está justo debajo quedan a 32 posiciones de distancia en el vector, y la red trataría igual cualquier permutación fija de los píxeles. Por eso no puede detectar directamente patrones locales como bordes, esquinas o texturas, ni reutilizar lo aprendido en una región en otra parte de la imagen. Si el mismo gato se desplaza unos píxeles, el vector cambia por completo y la red lo ve como un ejemplo distinto (no hay invariancia a traslaciones). Además cada neurona de la primera capa necesita un peso por píxel, lo que da muchos parámetros (131 200 solo en esa capa).

### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial de las imágenes?

**Respuesta:**

Las **redes neuronales convolucionales (CNN)**. Usan filtros pequeños que se deslizan por la imagen con los mismos pesos en todas las posiciones, así que aprovechan la vecindad entre píxeles, detectan patrones locales donde sea que aparezcan y necesitan muchos menos parámetros. Con capas de pooling y varias capas convolucionales apiladas construyen jerarquías: bordes, luego formas, luego partes de objetos. En CIFAR10 una CNN sencilla supera con facilidad el 70 % de accuracy.
---

## Para pensar

Nuestro modelo hace:

```text
32×32 → Flatten → 1024 valores → Dense → ReLU → Dense(10)
```

Sin embargo, una imagen tiene **estructura espacial**: existen bordes, formas y patrones locales.

Este resultado nos deja preparada la siguiente pregunta del curso:

> **¿Cómo puede una red aprovechar directamente la estructura espacial de una imagen?**

La respuesta será: **Convolutional Neural Networks (CNNs)**.
